In [17]:
#A. upload raw step1 json file 
import json
from datasets import Dataset

with open('/home/mjgwak/workspace/search-o1-dev/scripts/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.14,18:29-5.json', 'r') as f:
    step1_full_raw = json.load(f)

dataset = Dataset.from_list(step1_full_raw)
dataset.push_to_hub('talzoomanzoo/0514-step1-full-raw', private=False)

FileNotFoundError: [Errno 2] No such file or directory: '/home/mjgwak/workspace/search-o1-dev/scripts/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.14,18:29-5.json'

In [4]:
#B. upload raw step2 json file 
import json
from datasets import Dataset

with open('/home/mjgwak/workspace/search-o1-dev/scripts/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.14,18:29-5.json', 'r') as f:
    step2_full_raw = json.load(f)

dataset = Dataset.from_list(step2_full_raw)
dataset.push_to_hub('talzoomanzoo/0514-step2-full-raw', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.06s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0514-step2-full-raw/commit/69384c6ee8728fb65238fbd065de552d87b3c933', commit_message='Upload dataset', commit_description='', oid='69384c6ee8728fb65238fbd065de552d87b3c933', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0514-step2-full-raw', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0514-step2-full-raw'), pr_revision=None, pr_num=None)

In [5]:
#check full dataset len
import json
with open('/home/mjgwak/workspace/search-o1-dev/scripts/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.14,18:29-5.json', 'r') as f:
    full_data = json.load(f)
len(full_data) #993 x 5 x 5

23325

In [17]:
#C. RFT for y
import json

with open('/home/mjgwak/workspace/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.14,18:29-5.json', 'r') as f:
    sft_data = json.load(f)

rft_y_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

count = 0
for data in sft_data:
    count += 1
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    if has_math_equal == True:
        entry = {
            'idx': count,
            'id': data.get('new_id', None),
            'input_x': '',
            'ground_truth': '',
            'output_z': '',
            'output_y': '',
            'output_z_y': '',
            'metrics': []
        }

        # Collect all matching fields
        for key in data:
            if key.startswith('Question_'):
                entry['input_x'] = extract_x(data[key])
                entry['output_z'] = "<think>" + extract_z(data[key])
            elif key.startswith('Answer_'):
                entry['ground_truth'] = data[key]
            elif key.startswith('Output_'):
                entry['output_y'] = "</think>" + data[key]
            elif key.startswith('Metrics_'):
                entry['metrics'].append(data[key])
            entry['output_z_y'] = entry['output_z'] + entry['output_y']
        
        rft_y_data.append(entry)

with open('0514.rft_y_data.json', 'w') as f:
    json.dump(rft_y_data, f, ensure_ascii=False, indent=4)
print(len(rft_y_data))

9481


In [18]:
#huggingface push
from datasets import Dataset

dataset = Dataset.from_list(rft_y_data)
dataset.push_to_hub('talzoomanzoo/0514-rft-y-data', private=False)

Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 24.32ba/s]
/home/mjgwak/miniconda3/envs/reasoning/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(
Uploading the dataset shards: 100%|██████████| 1/1 [00:08<00:00,  8.82s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0514-rft-y-data/commit/09b35478db64b2cbfad41aaa83c0cbd578ba5c7f', commit_message='Upload dataset', commit_description='', oid='09b35478db64b2cbfad41aaa83c0cbd578ba5c7f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0514-rft-y-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0514-rft-y-data'), pr_revision=None, pr_num=None)

In [10]:
#3. dpo, rft, and EMDR^2 for z
# x, z_chosen, y_chosen under z _chosen, y_reject under z _chosen, z_reject, y_chosen under z _reject, y_reject under z _reject 
import json
import re

with open('/home/mjgwak/workspace/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.14,18:29-5.json', 'r') as f:
    z_y_combined_data = json.load(f)

filtered_z_y_combined_data = []

# Process data in chunks of 25 rows
for idx in range(0, len(z_y_combined_data), 25):
    chunk_25 = z_y_combined_data[idx:idx+25]
    subgroup_data = []  # Store (percentage, data) pairs for each subgroup
    subgroup_z_chosen_y_chosen = []
    subgroup_z_chosen_y_rejected = []
    subgroup_z_rejected_y_chosen = []
    subgroup_z_rejected_y_rejected = []
    
    # Process each subgroup of 5 rows within the 25-row chunk
    for i in range(0, len(chunk_25), 5):
        subgroup = chunk_25[i:i+5]
        true_pairs = []
        false_pairs = []
        
        # Count true and false pairs in this subgroup
        for data in subgroup:
            for key in data:
                if key.startswith('Metrics_'):
                    if data[key].get('math_equal', True):
                        true_pairs.append(data)
                    else:
                        false_pairs.append(data)
        
        # Calculate z_true_percentage for this subgroup
        z_true_percentage = len(true_pairs) / len(subgroup)
        
        # Store the percentage and corresponding data
        if 0.5 <= z_true_percentage < 1.0 and true_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'chosen'))
        elif 0 < z_true_percentage < 0.5 and false_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'rejected'))
    
    # If we have data, find the highest and lowest percentage entries
    if subgroup_data:
        # Sort by percentage
        subgroup_data.sort(key=lambda x: x[0])
        
        # Get the highest percentage (chosen) and lowest percentage (rejected)
        chosen_z_data = None
        rejected_z_data = None
        chosen_z_chosen_y_data = None
        chosen_z_rejected_y_data = None
        rejected_z_chosen_y_data = None
        rejected_z_rejected_y_data = None
        
        for percentage, true_data, false_data, type_ in subgroup_data:
            if type_ == 'chosen' and chosen_z_data is None:
                chosen_z_data = true_data
                chosen_z_chosen_y_data = true_data
                chosen_z_rejected_y_data = false_data
            elif type_ == 'rejected' and rejected_z_data is None:
                rejected_z_data = false_data
                rejected_z_chosen_y_data = true_data
                rejected_z_rejected_y_data = false_data
        
        # If we have both chosen and rejected data, create an entry
        if chosen_z_data and rejected_z_data:
            #x, z_chosen, y_chosen under z _chosen, y_reject under z _chosen, z_reject, y_chosen under z _reject, y_reject under z _reject 
            entry = {
                'idx': idx+1,
                'id': chosen_z_data.get('new_id', None).split('_')[0],
                'input_x': '',
                'chosen_z': '',
                'chosen_y_under_chosen_z': '',
                'rejected_y_under_chosen_z': '',
                'rejected_z': '',
                'chosen_y_under_rejected_z': '',
                'rejected_y_under_rejected_z': ''
            }
            
            # Process chosen z data
            for key in chosen_z_data:
                if key.startswith('Question_'):
                    entry['input_x'] = extract_x(chosen_z_data[key])
                    entry['chosen_z'] = "<think>" + extract_z(chosen_z_data[key])
                if key.startswith('Answer_'):
                    entry['ground_truth'] = chosen_z_data[key]
            for key in chosen_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_chosen_z'] = "</think>" + chosen_z_chosen_y_data[key]
            for key in chosen_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_chosen_z'] = "</think>" + chosen_z_rejected_y_data[key]
            
            # Process rejected z data
            for key in rejected_z_data:
                if key.startswith('Question_'):
                    entry['rejected_z'] = "<think>" + extract_z(rejected_z_data[key])
            for key in rejected_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_rejected_z'] = "</think>" + rejected_z_chosen_y_data[key]
            for key in rejected_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_rejected_z'] = "</think>" + rejected_z_rejected_y_data[key]
            
            filtered_z_y_combined_data.append(entry)

with open('0514.z_y_combined_data.json', 'w') as f:
    json.dump(filtered_z_y_combined_data, f, ensure_ascii=False, indent=4)

In [11]:
#huggingface push
from datasets import Dataset

dataset = Dataset.from_list(filtered_z_y_combined_data)
dataset.push_to_hub('talzoomanzoo/0514-z-y-combined-data', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.09s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0514-z-y-combined-data/commit/c69d56df410310906ffd1f340e6c872c5e0b9f5f', commit_message='Upload dataset', commit_description='', oid='c69d56df410310906ffd1f340e6c872c5e0b9f5f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0514-z-y-combined-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0514-z-y-combined-data'), pr_revision=None, pr_num=None)

In [19]:
#C. DPO for y
import json

with open('/home/mjgwak/workspace/search-o1-dev/src/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.14,18:29-5.json', 'r') as f:
    sft_data = json.load(f)

dpo_y_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

count = 0
for data in sft_data:
    count += 1
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    if has_math_equal == True:
        entry = {
            'idx': count,
            'id': data.get('new_id', None),
            'input_x': '',
            'ground_truth': '',
            'output_z': '',
            'output_y': '',
            'output_z_y': '',
            'metrics': []
        }

        # Collect all matching fields
        for key in data:
            if key.startswith('Question_'):
                entry['input_x'] = extract_x(data[key])
                entry['output_z'] = "<think>" + extract_z(data[key])
            elif key.startswith('Answer_'):
                entry['ground_truth'] = data[key]
            elif key.startswith('Output_'):
                entry['output_y'] = "</think>" + data[key]
            elif key.startswith('Metrics_'):
                entry['metrics'].append(data[key])
            entry['output_z_y'] = entry['output_z'] + entry['output_y']
        
        dpo_y_data.append(entry)
    
    if has_math_equal == False:
        entry = {
            'idx': count,
            'id': data.get('new_id', None),
            'input_x': '',
            'ground_truth': '',
            'output_z': '',
            'output_y': '',
            'output_z_y': '',
            'metrics': []
        }

        # Collect all matching fields
        for key in data:
            if key.startswith('Question_'):
                entry['input_x'] = extract_x(data[key])
                entry['output_z'] = "<think>" + extract_z(data[key])
            elif key.startswith('Answer_'):
                entry['ground_truth'] = data[key]
            elif key.startswith('Output_'):
                entry['output_y'] = "</think>" + data[key]
            elif key.startswith('Metrics_'):
                entry['metrics'].append(data[key])
            entry['output_z_y'] = entry['output_z'] + entry['output_y']
        
        dpo_y_data.append(entry)

with open('0514.dpo_y_data.json', 'w') as f:
    json.dump(dpo_y_data, f, ensure_ascii=False, indent=4)
print(len(dpo_y_data))

23325


In [20]:
#huggingface push
from datasets import Dataset

dataset = Dataset.from_list(dpo_y_data)
dataset.push_to_hub('talzoomanzoo/0514-dpo-y-data', private=False)

Uploading the dataset shards: 100%|██████████| 3/3 [00:27<00:00,  9.28s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0514-dpo-y-data/commit/4a82d04e3356388ea1cc860c07c2ef10a65a9806', commit_message='Upload dataset', commit_description='', oid='4a82d04e3356388ea1cc860c07c2ef10a65a9806', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0514-dpo-y-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0514-dpo-y-data'), pr_revision=None, pr_num=None)